### Building a function to parse recipes from websites

In [ ]:
import requests 
from bs4 import BeautifulSoup
import json

# Extract ingredient list from a recipe page using schema.org JSON-LD
def extract_ingredients(url):
    headers = {
    "User-Agent": "Chrome/91.0.4472.120"
    }
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")
    scripts = soup.find_all("script", type="application/ld+json")
    for script in scripts:
        if not script.string:
            continue
        if "recipeIngredient" not in script.string:
            continue
        try: 
            data = json.loads(script.string.strip())
        except json.JSONDecodeError:
            continue

        # handle simple Recipe JSON directly
        if isinstance(data, dict) and data.get("@type") == "Recipe": 
            ingredients = data.get("recipeIngredient", [])
            return {"Recipe_url": url, "Ingredients": ingredients}

        if isinstance(data, dict):
            candidates = data.get("@graph", [data])

        elif isinstance(data, list):
            candidates = data 
        
        else: 
            continue

        for item in candidates:
            if not isinstance(item, dict):
                continue
            
            ingredients = item.get("recipeIngredient", [])
            
            if ingredients:
                return {"Recipe_url": url, "Ingredients": ingredients}
         
    return None 


### Testing the parser

In [ ]:
test = extract_ingredients("https://ashbaber.com/easy-fudgy-blondies")
print(test)

### Exploring Spoonacular API as an alternative data source

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("SPOONACULAR_API_KEY")

search_url = "https://api.spoonacular.com/recipes/complexSearch"

queries = ["blondies", "classic blondies", "brown sugar blondies", "blondie bars", "fudgy blondies"]

all_recipes = []
seen_ids = set()  # To avoid duplicate recipes across queries

for query in queries:
    params = {
        "query": query,
        "number": 20,
        "apiKey": api_key
    }
    response = requests.get(search_url, params=params)
    recipes = response.json().get("results", [])
    print(f"Query '{query}': {len(recipes)} results")
    
    for recipe in recipes:
        if recipe["id"] not in seen_ids:
            seen_ids.add(recipe["id"])
            all_recipes.append(recipe)

print(f"Total unique recipes found: {len(all_recipes)}")


### Findings

Spoonacular returned only 1 unique blondie recipe across 5 different search queries, 
which is insufficient for meaningful analysis. The original JSON-LD scraper is retained 
as the primary data collection method.

### Making a dataframe (df) out of the parsed data

In [ ]:
import pandas as pd

df_urls = pd.read_csv("blondie_urls.csv")

df_urls.describe()

df_urls = df_urls.drop_duplicates(subset=["URLs"])
df_urls = df_urls.dropna(subset=["URLs"])

In [ ]:
urls = df_urls["URLs"].tolist()
print(len(urls))

In [ ]:
dataset = []
for url in urls:
    recipe = extract_ingredients(url)
    if recipe:
        dataset.append(recipe)
    else:
        print(f"Failed to extract from: {url}")

In [ ]:
print(len(dataset))

In [ ]:
df = pd.DataFrame(dataset)

df.describe()

In [ ]:
df["Ingredients"].sample(20)

### Exploding df — One row per ingredient

In [ ]:
df = df.explode("Ingredients").reset_index(drop=True)

In [ ]:
print(len(df))

### Making a "grams" column in df to show the weight of ingredients in grams

In [ ]:
def get_grams(text):
    import re
    match = re.search(r"(\d+\.?\d*)\s*g", text.lower())
    if match:
        return float(match.group(1))
    return None

In [ ]:
df["grams"] = df["Ingredients"].apply(get_grams)

In [ ]:
df.sample(10)

### Normalising fractions in list of ingredients

In [ ]:
df[df["Ingredients"].str.contains("⅕|⅖|⅗|⅘",na=False)]

In [ ]:
df[df["Ingredients"].str.contains("11/2",na=False)]

In [ ]:
def normalise_fractions(text):
    import re
    
    # Step 1: add space between number and unicode fraction
    text = re.sub(r"(\d)([½¼¾⅓⅔])", r"\1 \2", text)

    replacements = {
        "½": "1/2",
        "¼": "1/4",
        "¾": "3/4",
        "⅓": "1/3",
        "⅔": "2/3"}
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

In [ ]:
df["Ingredients"] = df["Ingredients"].apply(normalise_fractions)

In [ ]:
df[df["Ingredients"].str.contains("½|¼|¾|⅓|⅔",na=False)]

### Ensuring compact fractions are fixed (eg: 13/4 is computed as 1 and 3/4 instead of 13/4)

In [ ]:
def fix_compact_fractions(text):
    import re
    text = re.sub(r"\b(\d)(\d)/(\d)\b", r"\1 \2/\3", text)
    return text

In [ ]:
df["Ingredients"] = df["Ingredients"].apply(fix_compact_fractions)

In [ ]:
df[df["Ingredients"].str.contains("13/4",na=False)]

In [ ]:
df[df["Ingredients"].str.contains("11/2",na=False)]

### Extracts the first numeric amount from a text string; handles mixed numbers(1 1/2), fractions (3/4) and integers/decimals(2 or 2.5)

In [ ]:
def get_amount(text):
    import re
    match = re.search(r"(\d+\s\d+/\d+|\d+/\d+|\d+\.?\d*)", text)
    if match:
        return match.group(1)
    return None

df["amount_raw"] = df["Ingredients"].apply(get_amount)

### Converting the numeric amount into float

In [ ]:
def amount_to_float(text):
    if text is None:
        return None
    # case 1: Mixed number (e.g., "1 1/2")
    if " " in text:
        whole, frac = text.split(" ")
        num, den = frac.split("/")
        return float(whole) + float(num) / float(den)
    # case 2: Simple fraction (e.g., "1/2")
    if "/" in text:
        num, den = text.split("/")
        return float(num) / float(den)
    # case 3: Simple number (e.g., "1", "1.5")
    return float(text)

In [ ]:
df["amount_float"] = df["amount_raw"].apply(amount_to_float)

In [ ]:
df.sample(10)

In [ ]:
df[df["Ingredients"].str.contains("gram", na=False)]

In [ ]:
df[df["Recipe_url"].str.contains("ashbaber", na=False)]

### Extracting unit of measurement of each ingredient

In [ ]:
def get_unit(text):
    import re
    match = re.search(r"\b(cups|cup|c\.|tablespoons|tablespoon|teaspoons|teaspoon|tbsp|tsps|tsp|grams|gram|g|sticks|stick|ounces|ounce|oz)", text.lower())
    if match:
        unit = match.group(1)
        if unit == "c.":
            return "cup"
        return unit
    return None

df["unit"] = df["Ingredients"].apply(get_unit)

In [ ]:
df.sample(20)

In [ ]:
df[df["Ingredients"].str.contains("egg", case=False, na=False)][["Ingredients", "unit"]]

In [ ]:
df[df["Ingredients"].str.contains("cornstarch", case=False, na=False)]

### Assigning an "Ingredient type" for ingredients

In [ ]:
def get_ingredient_type(text):
    import re

    text = text.lower()
    keywords = ["butter", "flour", "sugar", "egg", "vanilla"]

    for word in keywords:
        if re.search(rf"\b{word}\b", text):
            return word
    if re.search(r"baking powder|baking soda", text):
        return "baking_powder"
    elif re.search(r"cornstarch", text):
        return "cornstarch"
    elif re.search(r"salt", text):
        return "salt"
    else:
        return "add_ins"

In [ ]:
df["ingredient_type"] = df["Ingredients"].apply(get_ingredient_type)

In [ ]:
df[df["ingredient_type"] == "add_ins"]

In [ ]:
df.sample(20)

In [ ]:
df[df["ingredient_type"].str.contains("flour", case=False, na=False)][["ingredient_type", "grams", "unit"]]

### Converting all units of measurement into grams for ease of analysis

In [ ]:
def convert_to_grams(row):

    if pd.notna(row["grams"]):
        return row["grams"]
    
    ingredient = row["ingredient_type"]
    amount = row["amount_float"]
    unit = row["unit"]

    cup = ["cup", "cups"]
    tbsp = ["tablespoons", "tablespoon", "tbsp"]
    tsp = ["teaspoons", "teaspoon", "tsps", "tsp"]
    oz = ["oz", "ounce", "ounces"]

    if amount is None:
        return None
    
    # Butter
    if ingredient == "butter":
        if unit in cup:
            return amount * 227
        elif unit in tbsp:
            return amount * 14
        elif unit in tsp:
            return amount * 5
        elif unit in oz:
            return amount * 28.35
        elif unit in ["sticks", "stick"]:
            return amount * 113

    # Flour
    elif ingredient == "flour":
        if unit in cup:
            return amount * 120
        elif unit in tbsp:
            return amount * 8
        elif unit in tsp:
            return amount * 3
        elif unit in oz:
            return amount * 28.35
        
    # Sugar
    elif ingredient == "sugar":
         if unit in cup:
            return amount * 200
         elif unit in tbsp:
            return amount * 12.5
         elif unit in tsp:
            return amount * 4
         elif unit in oz:
            return amount * 28.35
         
    # Vanilla
    elif ingredient == "vanilla":
        if unit in tsp:
            return amount * 5   # 1 tsp ≈ 5g/ml
        elif unit in tbsp:
            return amount * 15  # 1 tbsp = 3 tsp
        
    # Baking Powder
    elif ingredient == "baking_powder":
        if unit in tsp:
            return amount * 4   # 1 tsp ≈ 4g
        elif unit in tbsp:
            return amount * 12  # 1 tbsp = 3 tsp

    # Cornstarch
    elif ingredient == "cornstarch":
        if unit in tsp:
            return amount * 3   # 1 tsp ≈ 3g
        elif unit in tbsp:
            return amount * 9   # 1 tbsp = 3 tsp
        elif unit in cup:
            return amount * 120 # 1 cup ≈ 120g         
    return None

In [ ]:
df.head(10).apply(convert_to_grams, axis=1)

In [ ]:
df["grams_converted"] = df.apply(convert_to_grams, axis=1)

In [ ]:
df[df["Ingredients"].str.contains("baking powder", na=False)]

### Pivoting to one row per recipe for ratio analysis

In [ ]:
df_clean = df[df["grams_converted"].notna()]

df_grouped = df_clean.groupby(["Recipe_url", "ingredient_type"])["grams_converted"].sum().reset_index()

In [ ]:
df_pivot = df_grouped.pivot(index="Recipe_url", columns="ingredient_type", values="grams_converted")

df_pivot.sample(10)

In [ ]:
df_pivot = df_pivot.dropna(subset=["butter", "flour", "sugar"])

### Calculating ratios of the three main ingredient_types against each other

In [ ]:
df_pivot["butter_to_flour"] = df_pivot["butter"] / df_pivot["flour"]
df_pivot["butter_to_sugar"] = df_pivot["butter"] / df_pivot["sugar"]
df_pivot["sugar_to_flour"] = df_pivot["sugar"] / df_pivot["flour"]


In [ ]:
df_pivot.sample()

In [ ]:
df_pivot["butter_to_flour"].describe()

In [ ]:
df_pivot["butter_to_sugar"].describe()

In [ ]:
df_pivot["sugar_to_flour"].describe()

### Ingredient Calculator v1

In [ ]:
def blondie_ingredient_calculator(flour=None, butter=None, sugar=None):

    """
    Calculate blondie ingredient amounts based on one input ingredient.
    
    Provide ONLY ONE of the following:
    - flour (in grams)
    - butter (in grams)
    - sugar (in grams)
    
    Returns:
        dict with flour, butter, and sugar in grams
    """

    butter_to_flour = df_pivot["butter_to_flour"].median()
    butter_to_sugar = df_pivot["butter_to_sugar"].median()
    sugar_to_flour = df_pivot["sugar_to_flour"].median()

    # --- Input validation ---
    inputs_provided = sum(x is not None for x in [flour, butter, sugar])
    if inputs_provided == 0:
        return "Please provide one ingredient."
    if inputs_provided > 1:
        return "Please provide ONLY one ingredient at a time."
    
    # --- Core logic ---
    if flour is not None:
        butter = flour * butter_to_flour
        sugar = flour * sugar_to_flour
    elif butter is not None:
        flour = butter / butter_to_flour
        sugar = butter / butter_to_sugar
    elif sugar is not None:
        flour = sugar / sugar_to_flour
        butter = sugar * butter_to_sugar

    # --- Output ---
    return {
        "flour_g": round(flour, 1), "butter_g": round(butter, 1), "sugar_g": round(sugar, 1)
    }
    

In [ ]:
blondie_ingredient_calculator(butter = 100)

### Making a separate df for eggs

In [ ]:
df_eggs = df[df["ingredient_type"] == "egg"]

In [ ]:
df_eggs.head()

### Extracting the amount of whole eggs and egg yolks for analysis

In [ ]:
def extract_egg_info(text):
    import re
    text = text.lower()

    # normalize connectors
    text = re.sub(r"\b(plus|and)\b", " ", text)
    text = re.sub(r"[+,]", " ", text)

    whole_eggs = 0
    egg_yolks = 0

    # find all egg-related phrases
    for match in re.finditer(r"(\d+)[^\d]*egg(?: yolk)?", text):
        number = int(match.group(1))
        phrase = match.group(0)

        if "yolk" in phrase:
            egg_yolks += number
        else:
            whole_eggs += number

    return whole_eggs, egg_yolks

In [ ]:
extract_egg_info("2 extremely very large eggs + 1 egg yolk")

In [ ]:
extract_egg_info("1 large egg plus 1 egg yolk, at room temperature")

In [ ]:
df[["whole_eggs", "egg_yolks"]] = df["Ingredients"].apply(lambda x: pd.Series(extract_egg_info(x)))

In [ ]:
df["whole_eggs"].value_counts()

In [ ]:
df.sample(20)

In [ ]:
# --- Unnecessary to make this new df, but always use .all(axis=1) when applying a condition to multiple columns of a df to ensure the condition is applied horizontally (row-wise)---
df_clean_eggs = df[df[["whole_eggs", "egg_yolks"]].notna().all(axis=1)]

In [ ]:
df_grouped_w_eggs = df.groupby(["Recipe_url", "ingredient_type"])[["whole_eggs", "egg_yolks"]].sum().reset_index()

In [ ]:
df_grouped_w_eggs.sample(30)

In [ ]:
df_eggs_w_recipe = df.groupby("Recipe_url")[["whole_eggs", "egg_yolks"]].sum().reset_index()

In [ ]:
df_eggs_w_recipe.head()

### Merging egg data into the main dataframe

In [ ]:
df_pivot = df_pivot.merge(df_eggs_w_recipe, on="Recipe_url", how="left")

In [ ]:
df_pivot.sample(10)

### Got sidetracked; checked for correlation of cornstrach and baking powder with other dry ingredients

In [ ]:
df_pivot.groupby("cornstarch")[["flour", "baking_powder"]].mean()

In [ ]:
df_pivot.groupby("baking_powder")["flour"].describe()

### Analyzing correlations between eggs and flour, sugar and butter

In [ ]:
df_pivot.groupby("whole_eggs")[["flour", "sugar", "butter"]].mean()

In [ ]:
df_pivot.groupby("whole_eggs")[["flour", "sugar", "butter"]].median()

In [ ]:
df_pivot.groupby("egg_yolks")[["flour", "sugar", "butter"]].mean()

In [ ]:
df_pivot.groupby("egg_yolks")[["flour", "sugar", "butter"]].median()

In [ ]:
df_pivot["fat_ratio"] = df_pivot["butter"]/df_pivot["flour"]
df_pivot["sugar_ratio"] = df_pivot["sugar"]/df_pivot["flour"]


In [ ]:
df_pivot.sample(10)

### Checking for correlation between whole eggs and fat_ratio and sugar_ratio

In [ ]:
df_pivot.groupby("whole_eggs")[["fat_ratio", "sugar_ratio"]].mean()

In [ ]:
df_pivot.groupby("whole_eggs")[["fat_ratio", "sugar_ratio"]].median()

### Checking for correlation between egg yolks and fat_ratio and sugar_ratio

In [ ]:
df_pivot.groupby("egg_yolks")[["fat_ratio", "sugar_ratio"]].mean()

In [ ]:
df_pivot.groupby("egg_yolks")[["fat_ratio", "sugar_ratio"]].median()

### Estimating the number of eggs

Base estimate: 1 egg per 100g flour, clamped between 1 and 4.
Adjusted up if fat ratio < 0.8, down if fat ratio > 1.05.

In [ ]:
def estimate_eggs(flour, butter):

    fat_ratio = butter / flour

    # --- Base estimate (size-driven) ---
    eggs = int(flour / 100)
    eggs = max(1, min(4, eggs))

    # --- Adjust using fat ratio ---
    if fat_ratio < 0.8:
        eggs += 1
    elif fat_ratio > 1.05:
        eggs -= 1

    # --- Clamp again ---
    eggs = max(1, min(4, eggs))

    return eggs

### Ingredient Calculator v2

With the included estimate_eggs function.

In [ ]:
def blondie_ingredient_calculator_v2(flour=None, butter=None, sugar=None):

    butter_to_flour = df_pivot["butter_to_flour"].median()
    sugar_to_flour = df_pivot["sugar_to_flour"].median()

    inputs_provided = sum(x is not None for x in [flour, butter, sugar])

    if inputs_provided == 0:
        return "Please provide one ingredient."
    if inputs_provided > 1:
        return "Please provide ONLY one ingredient at a time."

    # --- Convert everything to flour ---
    if butter is not None:
        flour = butter / butter_to_flour
    elif sugar is not None:
        flour = sugar / sugar_to_flour

    # --- Recompute consistently ---
    butter = flour * butter_to_flour
    sugar = flour * sugar_to_flour

    # --- Eggs ---
    eggs = estimate_eggs(flour, butter)

    return {
        "flour_g": round(flour, 1),
        "butter_g": round(butter, 1),
        "sugar_g": round(sugar, 1),
        "eggs": eggs
    }

In [ ]:
blondie_ingredient_calculator_v2(butter=200)

### Analysing brown vs white sugar ratios

In [ ]:
def get_sugar_type(text):
    import re

    text = text.lower()

    # --- brown sugar ---
    if re.search(r"\b(?:light\s+|dark\s+)?brown(?:\s+\w+)*\s+sugar\b", text):
        return "brown"

    # --- white sugar types ---
    elif re.search(r"\b(?:granulated|caster|white|powdered)\s+sugar\b", text):
        return "white"

    # --- generic sugar ---
    elif re.search(r"\bsugar\b", text):
        return "white"

    return None

In [ ]:
df_sugar = df[df["ingredient_type"] == "sugar"].copy()

df_sugar["sugar_type"] = df_sugar["Ingredients"].apply(get_sugar_type)

In [ ]:
df_sugar_grouped = df_sugar.groupby(
    ["Recipe_url", "sugar_type"]
)["grams_converted"].sum().reset_index()

In [ ]:
df_sugar_grouped.sample(10)

In [ ]:
df_sugar_pivot = df_sugar_grouped.pivot(
    index="Recipe_url",
    columns="sugar_type",
    values="grams_converted"
).fillna(0)

In [ ]:
df_mix = df_sugar_pivot[df_sugar_pivot["white"] > 0].copy()

In [ ]:
df_mix["brown/white"] = df_mix["brown"] / df_mix["white"]

In [ ]:
df_mix.sort_values(by="white")

In [ ]:
df_mix["brown"].median()

In [ ]:
df_pivot = df_pivot.merge(df_sugar_pivot, on="Recipe_url", how="left")

In [ ]:
df_pivot.sample(10)

In [ ]:
df_pivot[["sugar", "brown", "white"]].describe()

In [ ]:
df_pivot["brown_ratio"] = df_pivot["brown"] / df_pivot["sugar"]
df_pivot["white_ratio"] = df_pivot["white"] / df_pivot["sugar"]


In [ ]:
df_pivot["brown_ratio"].describe()

In [ ]:
df_pivot["white_ratio"].describe()

In [ ]:
df_sugar_pivot["total"] = df_sugar_pivot["brown"] + df_sugar_pivot["white"]

df_sugar_pivot[["total", "brown", "white"]].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df_mix["white"], df_mix["brown"])
plt.xlabel("White Sugar")
plt.ylabel("Brown Sugar")
plt.title("White sugar vs Brown sugar relation")
plt.show()

### Should white sugar be used for the final model?

White sugar only appears past the 75th percentile, making it a minority choice. The model defaults to all brown sugar.

### Analyzing vanilla ratios

In [ ]:
df_vanilla = df[df["ingredient_type"] == "vanilla"]

In [ ]:
df_vanilla["grams_converted"].isna().sum()

In [ ]:
df_vanilla_grouped = df_vanilla.groupby("Recipe_url")["grams_converted"].sum().reset_index()

df_vanilla_ratios = df_vanilla_grouped.merge(df_pivot[["Recipe_url", "flour"]], on="Recipe_url")

In [ ]:
df_vanilla_ratios.sample(20)

In [ ]:
df_vanilla_ratios["vanilla_to_flour"] = df_vanilla_ratios["grams_converted"] / df_vanilla_ratios["flour"]

In [ ]:
df_vanilla_ratios["vanilla_to_flour"].describe()

In [ ]:
#---Convert vanilla extract values from g/mL to teaspoons and tablespoons for ease of use---

def format_vanilla(vanilla_ml):

    tsp = vanilla_ml / 5
    tsp_rounded = round(tsp)

    tbsp = tsp_rounded // 3
    remaining_tsp = tsp_rounded % 3

    if tbsp > 0 and remaining_tsp > 0:
        return f"{tbsp} tbsp + {remaining_tsp} tsp"
    elif tbsp > 0:
        return f"{tbsp} tbsp"
    else:
        return f"{remaining_tsp} tsp"

### Analyzing baking powder ratios

In [ ]:
df_baking_powder = df[df["ingredient_type"] == "baking_powder"]

In [ ]:
df_baking_powder_grouped = df_baking_powder.groupby("Recipe_url")[["grams_converted"]].sum().reset_index()

df_baking_powder_ratios = df_baking_powder_grouped.merge(df_pivot[["Recipe_url", "flour"]], on="Recipe_url", how="left")


In [ ]:
df_baking_powder_ratios["bp_to_flour"] = df_baking_powder_ratios["grams_converted"] / df_baking_powder_ratios["flour"]

In [ ]:
df_baking_powder_ratios["bp_to_flour"].describe()

In [ ]:
df_baking_powder_ratios.sample(20)

In [ ]:
df[df["ingredient_type"] == "baking_powder"]["Recipe_url"].nunique()

In [ ]:
df_baking_powder_ratios.groupby("grams_converted")["flour"].max()

In [ ]:
def format_baking_powder(bp_g):
    tsp = bp_g / 4
    tsp_rounded = round(tsp * 4) / 4  # round to nearest 1/4 tsp

    formats = {
        0.25: "1/4 tsp",
        0.5: "1/2 tsp",
        0.75: "3/4 tsp",
        1.0: "1 tsp"
    }

    if tsp_rounded in formats:
        return formats[tsp_rounded]
    else:
        return f"{tsp_rounded:g} tsp"

### Saving processed data

Saving the key dataframes to CSV for reproducibility. Load these directly to skip the scraping and cleaning pipeline.

In [ ]:
df_pivot.to_csv("df_pivot.csv", index=False)
df_vanilla_ratios.to_csv("df_vanilla_ratios.csv", index=False)
df_baking_powder_ratios.to_csv("df_baking_powder_ratios.csv", index=False)

### Ingredient Calculator v3

The final calculator: takes one ingredient as input and returns a complete, scaled ingredient list with vanilla, baking powder, eggs, and instructions.

In [ ]:
def blondie_ingredient_calculator(flour=None, butter=None, sugar=None):

    butter_to_flour = df_pivot["butter_to_flour"].median()
    sugar_to_flour = df_pivot["sugar_to_flour"].median()
    vanilla_to_flour = df_vanilla_ratios["vanilla_to_flour"].median()
    baking_powder_to_flour = df_baking_powder_ratios["bp_to_flour"].median()

    inputs_provided = sum(x is not None and x !=0 for x in [flour, butter, sugar])

    if inputs_provided == 0:
        return "Please enter an amount greater than 0 to generate your recipe."
    if inputs_provided > 1:
        return "Please provide ONLY one ingredient at a time."
    if (flour is not None and flour < 0) or (butter is not None and butter < 0) or (sugar is not None and sugar < 0):
        return "Please enter a positive amount to generate your recipe."

    # --- Convert everything to flour ---
    if butter is not None and butter != 0:
        flour = butter / butter_to_flour
    elif sugar is not None and sugar != 0:
        flour = sugar / sugar_to_flour

    # --- Recompute consistently ---
    butter = flour * butter_to_flour
    sugar = flour * sugar_to_flour

    # --- Vanilla ---
    vanilla_ml = flour * vanilla_to_flour
    vanilla_text = format_vanilla(vanilla_ml)

    # --- Baking Powder ---
    baking_powder_g = flour * baking_powder_to_flour
    baking_powder_text = format_baking_powder(baking_powder_g)

    # --- Eggs ---
    eggs = estimate_eggs(flour, butter)

    result = {
        "flour_g": round(float(flour), 1),
        "baking_powder": baking_powder_text,
        "butter_g": round(float(butter), 1),
        "brown_sugar_g": round(float(sugar), 1),
        "eggs": eggs,
        "vanilla_extract": vanilla_text,
        'optional': "Add 2 tsp cornstarch for extra tenderness, an extra egg yolk for maximum fudginess, or both!"
    }

    return result

In [ ]:
blondie_ingredient_calculator_v3(butter=150)

### Gradio UI and Recipe Generator

Wraps the calculator in a simple interface. Select an ingredient, enter the amount, get a recipe.

In [ ]:
import gradio as gr

def blondie_recipe_generator(ingredient, amount):
    if ingredient == "Flour":
        result = blondie_ingredient_calculator(flour=amount)
    elif ingredient == "Butter":
        result = blondie_ingredient_calculator(butter=amount)
    elif ingredient == "Sugar":
        result = blondie_ingredient_calculator(sugar=amount)

    if isinstance(result, str):
        return result
    
    ingredient_list = f"""
    Ingredients:
    - All-purpose Flour: {result['flour_g']}g
    - Butter (Salted or Unsalted): {result['butter_g']}g
    - Brown Sugar: {result['brown_sugar_g']}g
    - {'Eggs' if result['eggs'] > 1 else 'Egg'}: {result['eggs']}
    - Vanilla Extract: {result['vanilla_extract']}
    - Baking Powder: {result['baking_powder']}
    Optional: {result['optional']}
    """
    
    cooking_instructions = """
    Instructions:
    1. Preheat oven to 180°C (350°F). Line a baking pan with parchment paper.
    2. Melt the butter. After it has cooled down, mix with the sugar and vanilla extract until combined.
    3. Beat in the eggs one at a time until fully combined. If using an extra egg yolk, add it in with the eggs.
    4. In a separate container, sift all the dry ingredients together (flour, baking powder, and cornstarch if using).
    5. Fold the dry ingredients into the wet ingredients until just combined.
    6. Fold in chocolate chips, chopped baking chocolate, nuts, or any other add-ins if using. Get creative, folks!
    7. Pour into the pan and bake for 20-25 minutes or until the top is browned and a toothpick inserted one inch from the edge comes out with moist crumbs.
    8. Let the blondie cool at room temperature for 1-2 hours, then refrigerate for at least one hour before serving.
    9. Sprinkle flaky salt on top and enjoy!
    
    Notes:
    - If using unsalted butter, add 1/2 tsp of table salt to the dry ingredients in step 4.
    - Chocolate chips hold their shape in the oven. If you want pools of melted chocolate, use chopped baking chocolate instead. Use both for the best of both worlds!
    - Blondies traditionally use all brown sugar, which is responsible for the caramel flavour and browning. If you only have white sugar, that works too — you'll get a delicious blondie with a crinkly top, just without the caramel depth. Replacing some of the brown sugar with white gives you both the caramel flavour and the crinkly top. 
    """
    return ingredient_list + cooking_instructions

theme = gr.themes.Citrus(
    primary_hue=gr.themes.colors.amber,
    secondary_hue=gr.themes.colors.stone,
    neutral_hue=gr.themes.colors.slate,
    font=gr.themes.GoogleFont("Outfit"),
).set(
    body_background_fill="#F5F5F7",
    body_text_color="#1d1d1f"
)

with gr.Blocks(theme=theme) as demo:
    gr.Markdown("# Blondie Recipe Generator")
    gr.Markdown("Select an ingredient, enter the amount in grams, and get a complete blondie recipe scaled to your kitchen.")
    
    with gr.Row():
        ingredient = gr.Dropdown(label="Select Ingredient", choices=["Flour", "Butter", "Sugar"])
        amount = gr.Number(label="Enter Amount in grams", precision=1, value=None)
    
    submit = gr.Button("Generate Recipe")
    output = gr.Textbox(label="Your Blondie Recipe")
    
    submit.click(blondie_recipe_generator, inputs=[ingredient, amount], outputs=output)

demo.launch()